# Cross-Lingual Embedding Alignment (SQuAD ↔ UQA)

This notebook mirrors the paper’s workflow but uses SQuAD (English) and UQA (Urdu) data.
We run two experiments:
1) Alignment quality: cosine distance between parallel question pairs before/after Procrustes.
2) Cross-lingual retrieval: Urdu queries retrieving English contexts, Recall@{1,3,5} before/after alignment.


In [1]:
from __future__ import annotations

import os
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
!pip install faiss-cpu
import faiss
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

np.random.seed(42)
pd.set_option("display.max_colwidth", 160)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 88.3 MB/s eta 0:00:00:00:0100:01


2026-01-05 06:48:48.002026: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767595728.402760      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767595728.516303      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767595729.565234      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767595729.565279      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767595729.565282      55 computation_placer.cc:177] computation placer alr

In [16]:
load_dotenv()
EMBED_MODEL_NAME = os.getenv("RAG_EMBED_MODEL", "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
MAX_ROWS = int(os.getenv("RAG_MAX_ROWS", "-1"))
SQUAD_SPLIT = os.getenv("SQUAD_SPLIT", "train")
UQA_SPLIT = os.getenv("UQA_SPLIT", "train")
ALIGN_TRAIN_RATIO = float(os.getenv("ALIGN_TRAIN_RATIO", "0.15"))
ALIGN_RANDOM_BASELINE_N = int(os.getenv("ALIGN_RANDOM_BASELINE_N", "100"))
RETRIEVAL_TRAIN_RATIO = float(os.getenv("RETRIEVAL_TRAIN_RATIO", "0.7"))
RETRIEVAL_K_VALUES = [1, 3, 5]
SQUAD_FILES = {
    "train": "plain_text/train-00000-of-00001.parquet",
    "validation": "plain_text/validation-00000-of-00001.parquet",
}
UQA_FILES = {
    "train": "data/train-00000-of-00001-bac007e8ca719235.parquet",
    "validation": "data/validation-00000-of-00001-cf8a6960dcbb53ee.parquet",
}
def hf_path(dataset: str, rel_path: str) -> str:
    return f"hf://datasets/{dataset}/{rel_path}"


## Load SQuAD + UQA

We use English SQuAD and Urdu UQA (Urdu script). This follows the paper’s protocol but not the Roman Urdu dataset.


In [18]:
print(f"Embedding model: {EMBED_MODEL_NAME}")
print("Loading SQuAD + UQA splits from Hugging Face hub...")

squad_path = hf_path("rajpurkar/squad", SQUAD_FILES[SQUAD_SPLIT])
uqa_path = hf_path("uqa/UQA", UQA_FILES[UQA_SPLIT])

squad_raw = pd.read_parquet(squad_path)
uqa_raw = pd.read_parquet(uqa_path)

print(
    f"Loaded SQuAD split='{SQUAD_SPLIT}' -> {len(squad_raw):,} rows, "
    f"UQA split='{UQA_SPLIT}' -> {len(uqa_raw):,} rows"
)


Embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Loading SQuAD + UQA splits from Hugging Face hub...
Loaded SQuAD split='train' -> 87,599 rows, UQA split='train' -> 124,745 rows


## Merge overlapping question IDs and cache

This creates a bilingual table with matched question IDs.


In [5]:
ID_CANDIDATES = ["id", "question_id", "qas_id", "guid", "uuid", "example_id"]

def pick_id_column(frame: pd.DataFrame, label: str) -> str:
    for candidate in ID_CANDIDATES:
        if candidate in frame.columns:
            return candidate
    raise KeyError(f"Could not find an ID column for {label}; checked {ID_CANDIDATES}")

squad_id_col = pick_id_column(squad_raw, "SQuAD")
uqa_id_col = pick_id_column(uqa_raw, "UQA")

overlap_ids = set(squad_raw[squad_id_col].astype(str)) & set(uqa_raw[uqa_id_col].astype(str))

merged = (
    squad_raw[squad_raw[squad_id_col].astype(str).isin(overlap_ids)]
    .merge(
        uqa_raw[uqa_raw[uqa_id_col].astype(str).isin(overlap_ids)],
        left_on=squad_id_col,
        right_on=uqa_id_col,
        suffixes=("_squad", "_uqa"),
    )
)

MERGED_CSV_PATH = Path(os.getenv("SQUAD_CSV_PATH", "Squad.csv")).resolve()
merged.to_csv(MERGED_CSV_PATH, index=False)
print(f"Merged pairs: {len(merged):,} -> {MERGED_CSV_PATH}")


Merged pairs: 83,018 -> /kaggle/working/Squad.csv


In [6]:
merged = pd.read_csv("Squad.csv")
merged.shape

(83018, 11)

## Build bilingual QA dataframe

We keep question + context fields for each language.


In [7]:
def build_language_records(frame: pd.DataFrame) -> pd.DataFrame:
    records: List[Dict[str, Any]] = []
    for idx, row in enumerate(frame.to_dict("records")):
        base_id = str(row.get("id", f"pair_{idx}")) or f"pair_{idx}"

        question_en = row.get("question_squad")
        context_en = row.get("context_squad")
        if isinstance(question_en, str) and question_en.strip() and isinstance(context_en, str) and context_en.strip():
            records.append(
                {
                    "pair_id": base_id,
                    "question_id": f"{base_id}_en",
                    "question": question_en.strip(),
                    "context": context_en.strip(),
                    "dataset": "squad_en",
                    "language": "en",
                    "language_label": "English",
                }
            )

        question_ur = row.get("question_uqa")
        context_ur = row.get("context_uqa")
        if isinstance(question_ur, str) and question_ur.strip() and isinstance(context_ur, str) and context_ur.strip():
            records.append(
                {
                    "pair_id": base_id,
                    "question_id": f"{base_id}_ur",
                    "question": question_ur.strip(),
                    "context": context_ur.strip(),
                    "dataset": "uqa_urdu",
                    "language": "ur",
                    "language_label": "Urdu",
                }
            )

    return pd.DataFrame(records)

df = build_language_records(merged)
if MAX_ROWS > 0:
    df = df.head(MAX_ROWS).copy()

print(f"Unified dataframe rows: {len(df):,} (unique pairs: {df['pair_id'].nunique():,})")
df.head(2)


Unified dataframe rows: 166,036 (unique pairs: 83,018)


,pair_id,question_id,question,context,dataset,language,language_label
0,5733be284776f41900661182,5733be284776f41900661182_en,To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?,"Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the ...",squad_en,en,English
1,5733be284776f41900661182,5733be284776f41900661182_ur,کنواری مریم نے 1858 میں فرانس کے شہر لوردس میں کس کے سامنے ظہور کیا؟,آرکیٹیکچرل طور پر ، اسکول میں ایک کیتھولک کردار ہے۔ مین بلڈنگ کے سونے کے گنبد کے اوپری حصے میں ورجن مریم کا ایک سنہری مجسمہ ہے۔ فوری طور پر مین بلڈنگ کے سام...,uqa_urdu,ur,Urdu


## Unique context pairs and query pool

We build one English document per unique English context and its Urdu translation.


In [8]:
context_pairs = (
    merged
    .dropna(subset=["context_squad", "context_uqa"])
    .drop_duplicates(subset=["context_squad"])
    .loc[:, ["context_squad", "context_uqa"]]
    .reset_index(drop=True)
)
context_pairs["context_id"] = [f"ctx_{i}" for i in range(len(context_pairs))]

context_id_en = dict(zip(context_pairs["context_squad"], context_pairs["context_id"]))
context_id_ur = dict(zip(context_pairs["context_uqa"], context_pairs["context_id"]))

df["context_id"] = df.apply(
    lambda r: context_id_en.get(r.context) if r.language == "en" else context_id_ur.get(r.context),
    axis=1,
)

query_df = (
    df[df["language"] == "ur"]
    .dropna(subset=["context_id"])
    .drop_duplicates(subset=["context_id"])
    .loc[:, ["context_id", "question"]]
    .reset_index(drop=True)
)

print(f"Unique context pairs: {len(context_pairs):,}")
print(f"Unique Urdu queries (1 per context): {len(query_df):,}")


Unique context pairs: 18,857
Unique Urdu queries (1 per context): 18,828


## Build English document index

We index English contexts only, matching the paper's Roman Urdu -> English retrieval setup.


In [9]:
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

doc_df = context_pairs[["context_id", "context_squad"]].rename(
    columns={"context_squad": "doc_text"}
)

doc_embeddings = embed_model.encode(
    doc_df["doc_text"].tolist(),
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True,
).astype("float32")

doc_index = faiss.IndexFlatIP(doc_embeddings.shape[1])
doc_index.add(doc_embeddings)

print(
    f"Indexed {doc_index.ntotal:,} English documents (dim={doc_embeddings.shape[1]})"
)


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/295 [00:00<?, ?it/s]

Indexed 18,857 English documents (dim=384)


## Procrustes alignment helper
 RAG Assessment (RAGAS) evaluation

In [10]:
def assert_finite(name: str, arr: np.ndarray) -> None:
    if not np.isfinite(arr).all():
        raise ValueError(f"{name} contains NaN/inf values")

def safe_l2_normalize(vecs: np.ndarray, epsilon: float = 1e-9) -> np.ndarray:
    norms = np.linalg.norm(vecs, axis=1, keepdims=True)
    return vecs / (norms + epsilon)

def sanitize_embeddings(name: str, vecs: np.ndarray) -> np.ndarray:
    vecs = np.nan_to_num(vecs, nan=0.0, posinf=0.0, neginf=0.0)
    assert_finite(name, vecs)
    return vecs

def filter_empty_vectors(src: np.ndarray, tgt: np.ndarray, epsilon: float = 1e-9) -> Tuple[np.ndarray, np.ndarray]:
    src_norms = np.linalg.norm(src, axis=1)
    tgt_norms = np.linalg.norm(tgt, axis=1)
    mask = (src_norms > epsilon) & (tgt_norms > epsilon)
    return src[mask], tgt[mask]

def learn_procrustes_alignment(src_embeddings: np.ndarray, tgt_embeddings: np.ndarray, center: bool = True) -> Dict[str, Any]:
    src = np.asarray(src_embeddings, dtype=np.float64)
    tgt = np.asarray(tgt_embeddings, dtype=np.float64)
    if src.shape != tgt.shape:
        raise ValueError(f"Shape mismatch: src {src.shape} vs tgt {tgt.shape}")

    src = safe_l2_normalize(src)
    tgt = safe_l2_normalize(tgt)
    src = sanitize_embeddings("src_embeddings_norm", src)
    tgt = sanitize_embeddings("tgt_embeddings_norm", tgt)

    src, tgt = filter_empty_vectors(src, tgt)
    if len(src) == 0:
        raise ValueError("No valid vectors after filtering empty embeddings")

    if center:
        src_mean = src.mean(axis=0, keepdims=True)
        tgt_mean = tgt.mean(axis=0, keepdims=True)
        src_centered = src - src_mean
        tgt_centered = tgt - tgt_mean
    else:
        src_mean = np.zeros((1, src.shape[1]), dtype=src.dtype)
        tgt_mean = np.zeros((1, tgt.shape[1]), dtype=tgt.dtype)
        src_centered = src
        tgt_centered = tgt

    src_centered = sanitize_embeddings("src_centered", src_centered)
    tgt_centered = sanitize_embeddings("tgt_centered", tgt_centered)

    m = np.dot(src_centered.T, tgt_centered)
    assert_finite("cross_covariance", m)
    m = sanitize_embeddings("cross_covariance", m)

    try:
        u, _, vt = np.linalg.svd(m, full_matrices=False)
    except np.linalg.LinAlgError:
        m = m + np.eye(m.shape[0]) * 1e-12
        u, _, vt = np.linalg.svd(m, full_matrices=False)
    r = u @ vt

    if np.linalg.det(r) < 0:
        vt[-1, :] *= -1
        r = u @ vt

    r = sanitize_embeddings("alignment_matrix", r)
    return {"R": r.astype(np.float64), "src_mean": src_mean, "tgt_mean": tgt_mean, "center": center}

def apply_procrustes_alignment(embeddings: np.ndarray, alignment: Dict[str, Any]) -> np.ndarray:
    r = alignment["R"]
    assert_finite("R", r)
    emb = np.asarray(embeddings, dtype=np.float64)
    emb = sanitize_embeddings("input_embeddings", emb)
    emb = safe_l2_normalize(emb)
    if alignment.get("center"):
        emb = emb - alignment["src_mean"]
    aligned = emb @ r
    if alignment.get("center"):
        aligned = aligned + alignment["tgt_mean"]
    aligned = sanitize_embeddings("aligned_embeddings", aligned)
    return aligned.astype(np.float32)


#### Exp: Urdu context with English alignment.

## Sanity check embeddings

Verify embeddings are finite and norms look reasonable before alignment.


In [11]:
def describe_embeddings(label: str, emb: np.ndarray) -> None:
    emb = np.asarray(emb)
    norms = np.linalg.norm(emb, axis=1)
    print(label)
    print("  shape:", emb.shape)
    print("  finite:", bool(np.isfinite(emb).all()))
    print("  norm min/mean/max:", float(norms.min()), float(norms.mean()), float(norms.max()))

if "ur_train" in globals() and "en_train" in globals():
    describe_embeddings("UR train", ur_train)
    describe_embeddings("EN train", en_train)
else:
    print("Run Experiment 1 to create ur_train/en_train first.")


Run Experiment 1 to create ur_train/en_train first.


In [12]:
def cosine_distance(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    a = a / np.linalg.norm(a, axis=1, keepdims=True)
    b = b / np.linalg.norm(b, axis=1, keepdims=True)
    return 1.0 - np.sum(a * b, axis=1)

question_pairs = (
    df.pivot_table(index="pair_id", columns="language", values="question", aggfunc="first")
    .dropna(subset=["en", "ur"])
    .reset_index()
)

question_pairs = question_pairs.sample(frac=1.0, random_state=42).reset_index(drop=True)
cut = int(len(question_pairs) * ALIGN_TRAIN_RATIO)
train_q = question_pairs.iloc[:cut]
test_q = question_pairs.iloc[cut:]

en_train = embed_model.encode(train_q["en"].tolist(), convert_to_numpy=True, normalize_embeddings=True).astype("float32")
ur_train = embed_model.encode(train_q["ur"].tolist(), convert_to_numpy=True, normalize_embeddings=True).astype("float32")

assert_finite("ur_train", ur_train)
assert_finite("en_train", en_train)
alignment_q = learn_procrustes_alignment(ur_train, en_train, center=True)

en_test = embed_model.encode(test_q["en"].tolist(), convert_to_numpy=True, normalize_embeddings=True).astype("float32")
ur_test = embed_model.encode(test_q["ur"].tolist(), convert_to_numpy=True, normalize_embeddings=True).astype("float32")

dist_before = cosine_distance(ur_test, en_test).mean()
aligned_ur_test = apply_procrustes_alignment(ur_test, alignment_q)
dist_after = cosine_distance(aligned_ur_test, en_test).mean()

rand_n = min(ALIGN_RANDOM_BASELINE_N, len(en_test))
rand_idx = np.random.choice(len(en_test), size=rand_n, replace=False)
rand_en = en_test[rand_idx]
rand_ur = ur_test[rand_idx]
np.random.shuffle(rand_en)
dist_random = cosine_distance(rand_ur, rand_en).mean()

print({"random": round(float(dist_random), 4), "original": round(float(dist_before), 4), "aligned": round(float(dist_after), 4)})



{'random': 0.8563, 'original': 0.2048, 'aligned': 0.2041}


## Experiment 2: Cross-lingual retrieval (Recall@{1,3,5})

We use a 70/30 split over unique context pairs. Procrustes alignment is learned on 70% and evaluated on 30%.


In [13]:
def recall_at_k(ranks: List[Optional[int]], k_values: List[int]) -> Dict[str, float]:
    metrics: Dict[str, float] = {}
    for k in k_values:
        metrics[f"recall@{k}"] = float(np.mean([1.0 if r is not None and r <= k else 0.0 for r in ranks]))
    return metrics

RETRIEVAL_TRAIN_RATIO = float(os.getenv("RETRIEVAL_TRAIN_RATIO", "0.7"))
query_pool = query_df.sample(frac=1.0, random_state=42).reset_index(drop=True)
cut = int(len(query_pool) * RETRIEVAL_TRAIN_RATIO)
train_queries = query_pool.iloc[:cut].reset_index(drop=True)
test_queries = query_pool.iloc[cut:].reset_index(drop=True)

train_ids = train_queries["context_id"].tolist()
test_ids = test_queries["context_id"].tolist()

# Train alignment on query-document pairs
ur_train_q = embed_model.encode(
    train_queries["question"].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True,
).astype("float32")

doc_id_to_emb = {cid: doc_embeddings[i] for i, cid in enumerate(doc_df["context_id"].tolist())}
en_train_d = np.stack([doc_id_to_emb[cid] for cid in train_queries["context_id"].tolist()])

assert_finite("ur_train_q", ur_train_q)
assert_finite("en_train_d", en_train_d)
alignment_retrieval = learn_procrustes_alignment(ur_train_q, en_train_d, center=True)

# Build test-only document index
test_doc_mask = doc_df["context_id"].isin(test_ids).to_numpy()
test_doc_df = doc_df[test_doc_mask].reset_index(drop=True)
test_doc_embeddings = doc_embeddings[test_doc_mask]

test_index = faiss.IndexFlatIP(test_doc_embeddings.shape[1])
test_index.add(test_doc_embeddings)

def rank_query(query_text: str, true_context_id: str, alignment: Optional[Dict[str, Any]] = None) -> Optional[int]:
    qvec = embed_model.encode([query_text], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    if alignment is not None:
        qvec = apply_procrustes_alignment(qvec, alignment)
        qvec = safe_l2_normalize(qvec)
    scores, indices = test_index.search(qvec, test_index.ntotal)
    rank = 1
    for idx in indices[0]:
        if idx < 0:
            break
        if test_doc_df.iloc[idx]["context_id"] == true_context_id:
            return rank
        rank += 1
    return None

ranks_before: List[Optional[int]] = []
ranks_after: List[Optional[int]] = []

for row in tqdm(test_queries.itertuples(index=False), total=len(test_queries), desc="Retrieval eval"):
    ranks_before.append(rank_query(row.question, row.context_id, alignment=None))
    ranks_after.append(rank_query(row.question, row.context_id, alignment=alignment_retrieval))

print("Before alignment:", recall_at_k(ranks_before, RETRIEVAL_K_VALUES))
print("After alignment:", recall_at_k(ranks_after, RETRIEVAL_K_VALUES))





Retrieval eval: 100%|██████████| 5649/5649 [02:37<00:00, 35.78it/s]

Before alignment: {'recall@1': 0.3871481678173128, 'recall@3': 0.535670030093822, 'recall@5': 0.596388741370154}
After alignment: {'recall@1': 0.40591255089396355, 'recall@3': 0.5542573906886175, 'recall@5': 0.6149761019649496}


In [14]:
alignment_q = learn_procrustes_alignment(ur_train, en_train, center=True)
alignment_retrieval = learn_procrustes_alignment(ur_train_q, en_train_d, center=True)


## Build a per-query k-context pool

For a given query, include its gold context plus k-1 random contexts.


In [15]:
def sample_context_pool(context_id: str, *, k: Optional[int] = None, seed: int = 42) -> pd.DataFrame:
    if context_id not in set(doc_df["context_id"]):
        raise KeyError(f"context_id {context_id} not found in doc_df")

    pool = doc_df.copy()
    total = len(pool)
    if k is None:
        k = total
    if k < 1 or k > total:
        raise ValueError(f"k must be between 1 and {total}")

    gold = pool[pool["context_id"] == context_id]
    distractors = pool[pool["context_id"] != context_id]
    if len(distractors) < k - 1:
        raise ValueError("Not enough distractor contexts to sample")
    sampled = distractors.sample(k - 1, random_state=seed) if k > 1 else distractors.head(0)
    return pd.concat([gold, sampled], ignore_index=True)

def build_mini_index(context_pool: pd.DataFrame, *, model: Optional[SentenceTransformer] = None) -> Tuple[faiss.IndexFlatIP, np.ndarray]:
    if model is None:
        model = embed_model
    texts = context_pool["doc_text"].tolist()
    emb = model.encode(
        texts,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")
    idx = faiss.IndexFlatIP(emb.shape[1])
    idx.add(emb)
    return idx, emb

# Example usage (k=None means use all contexts)
example_context_id = test_queries.iloc[0]["context_id"]
mini_pool = sample_context_pool(example_context_id, k=None, seed=42)
mini_index, mini_embeddings = build_mini_index(mini_pool)
mini_pool.head(5)


,context_id,doc_text
0,ctx_14437,New chemical glass compositions or new treatment techniques can be initially investigated in small-scale laboratory experiments. The raw materials for labor...
1,ctx_11766,"Treatment of TB uses antibiotics to kill the bacteria. Effective TB treatment is difficult, due to the unusual structure and chemical composition of the myc..."
2,ctx_4590,"According to geologist Stefan Schmid, because the Western Alps underwent a metamorphic event in the Cenozoic Era while the Austroalpine peaks underwent an e..."
3,ctx_18729,Exhibitions and annual horse shows in all districts and a national horse and cattle show at Lahore are held with the official patronage. The national horse ...
4,ctx_7834,"The origins of the Ashkenazim are obscure, and many theories have arisen speculating about their ultimate provenance. The most well supported theory is the ..."


In [19]:
# Experiments + helpers (run this once before model-specific cells)
MODEL_CACHE: Dict[str, SentenceTransformer] = {}
MODEL_RESULTS: Dict[str, Dict[str, Any]] = {}

MODEL_CONFIG = {
    "MiniLM": {"model_name": EMBED_MODEL_NAME, "query_prefix": "", "doc_prefix": ""},
    "BGE": {"model_name": "BAAI/bge-m3", "query_prefix": "", "doc_prefix": ""},
    "IntFloat": {"model_name": "intfloat/multilingual-e5-large", "query_prefix": "query: ", "doc_prefix": "passage: "},
}

def get_model(model_name: str) -> SentenceTransformer:
    if model_name not in MODEL_CACHE:
        MODEL_CACHE[model_name] = SentenceTransformer(model_name)
    return MODEL_CACHE[model_name]

def encode_texts(model: SentenceTransformer, texts: List[str], *, prefix: str = "") -> np.ndarray:
    if prefix:
        texts = [prefix + t for t in texts]
    return model.encode(
        texts,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

def normalize_preview(text: str) -> str:
    text = text.replace("\n", " ")
    tokens = text.split()
    if tokens:
        single_ratio = sum(1 for t in tokens if len(t) == 1) / len(tokens)
        if single_ratio > 0.8:
            import re
            text = re.sub(r"\s{2,}", " <W> ", text)
            text = text.replace(" ", "")
            text = text.replace("<W>", " ")
    return " ".join(text.split())

def explain_gold_score(*, gold_rank: Optional[int], gold_score: Optional[float], top_score: float, k: int) -> None:
    if gold_rank is None:
        print("Gold context not in the candidate pool (rank=N/A).")
        return
    if gold_rank == 1:
        print("Gold context is top-1; score is already the highest.")
        return
    gap = top_score - (gold_score or 0.0)
    print(
        f"Gold context rank={gold_rank} (top-{k}). Score gap vs top-1: {gap:.4f}. "
        "This usually means the Urdu query is closer to another English context than its paired gold context "
        "(translation mismatch, paraphrase drift, or imperfect alignment)."
    )

def compute_alignment_quality(model: SentenceTransformer, *, label: str, query_prefix: str) -> None:
    en_train = encode_texts(model, train_q["en"].tolist(), prefix=query_prefix)
    ur_train = encode_texts(model, train_q["ur"].tolist(), prefix=query_prefix)
    ur_train, en_train = filter_empty_vectors(ur_train, en_train)
    if len(ur_train) == 0:
        print(f"{label} Experiment 1: no valid train vectors after filtering.")
        return
    alignment_q = learn_procrustes_alignment(ur_train, en_train, center=True)

    en_test = encode_texts(model, test_q["en"].tolist(), prefix=query_prefix)
    ur_test = encode_texts(model, test_q["ur"].tolist(), prefix=query_prefix)
    ur_test, en_test = filter_empty_vectors(ur_test, en_test)
    if len(ur_test) == 0:
        print(f"{label} Experiment 1: no valid test vectors after filtering.")
        return

    dist_before = cosine_distance(ur_test, en_test).mean()
    aligned_ur_test = apply_procrustes_alignment(ur_test, alignment_q)
    dist_after = cosine_distance(aligned_ur_test, en_test).mean()

    rand_n = min(ALIGN_RANDOM_BASELINE_N, len(en_test))
    rand_idx = np.random.choice(len(en_test), size=rand_n, replace=False)
    rand_en = en_test[rand_idx]
    rand_ur = ur_test[rand_idx]
    np.random.shuffle(rand_en)
    dist_random = cosine_distance(rand_ur, rand_en).mean()

    print(f"{label} Experiment 1 (cosine distance):", {
        "random": round(float(dist_random), 4),
        "original": round(float(dist_before), 4),
        "aligned": round(float(dist_after), 4),
    })

def compute_retrieval_metrics(
    model: SentenceTransformer,
    *,
    label: str,
    query_prefix: str,
    doc_prefix: str,
) -> Dict[str, Any]:
    ur_train_q = encode_texts(model, train_queries["question"].tolist(), prefix=query_prefix)
    doc_text_by_id = dict(zip(doc_df["context_id"], doc_df["doc_text"]))
    train_doc_texts = [doc_text_by_id[cid] for cid in train_queries["context_id"].tolist()]
    en_train_d = encode_texts(model, train_doc_texts, prefix=doc_prefix)

    ur_train_q, en_train_d = filter_empty_vectors(ur_train_q, en_train_d)
    if len(ur_train_q) == 0:
        print(f"{label} Experiment 2: no valid train vectors after filtering.")
        return {}
    alignment_retrieval = learn_procrustes_alignment(ur_train_q, en_train_d, center=True)

    test_doc_embeddings = encode_texts(model, test_doc_df["doc_text"].tolist(), prefix=doc_prefix)
    test_index = faiss.IndexFlatIP(test_doc_embeddings.shape[1])
    test_index.add(test_doc_embeddings)

    def rank_query(query_text: str, true_context_id: str, alignment: Optional[Dict[str, Any]] = None) -> Optional[int]:
        qvec = encode_texts(model, [query_text], prefix=query_prefix)
        if alignment is not None:
            qvec = apply_procrustes_alignment(qvec, alignment)
            qvec = safe_l2_normalize(qvec)
        scores, indices = test_index.search(qvec, test_index.ntotal)
        rank = 1
        for idx in indices[0]:
            if idx < 0:
                break
            if test_doc_df.iloc[idx]["context_id"] == true_context_id:
                return rank
            rank += 1
        return None

    ranks_before = []
    ranks_after = []
    for row in tqdm(test_queries.itertuples(index=False), total=len(test_queries), desc=f"{label} retrieval eval"):
        ranks_before.append(rank_query(row.question, row.context_id, alignment=None))
        ranks_after.append(rank_query(row.question, row.context_id, alignment=alignment_retrieval))

    print(f"{label} Experiment 2 (recall):")
    print("  Before alignment:", recall_at_k(ranks_before, RETRIEVAL_K_VALUES))
    print("  After alignment:", recall_at_k(ranks_after, RETRIEVAL_K_VALUES))

    return {
        "alignment_retrieval": alignment_retrieval,
        "model": model,
        "query_prefix": query_prefix,
        "doc_prefix": doc_prefix,
    }

def run_experiments_for_model(label: str) -> Dict[str, Any]:
    cfg = MODEL_CONFIG[label]
    model = get_model(cfg["model_name"])
    print(f"=== {label} ({cfg['model_name']}) ===")
    compute_alignment_quality(model, label=label, query_prefix=cfg["query_prefix"])
    result = compute_retrieval_metrics(
        model,
        label=label,
        query_prefix=cfg["query_prefix"],
        doc_prefix=cfg["doc_prefix"],
    )
    MODEL_RESULTS[label] = result
    return result

def search_and_report(
    *,
    model: SentenceTransformer,
    query_text: str,
    true_context_id: str,
    context_pool: pd.DataFrame,
    alignment: Optional[Dict[str, Any]],
    query_prefix: str,
    doc_prefix: str,
    k: int = 5,
) -> None:
    doc_emb = encode_texts(model, context_pool["doc_text"].tolist(), prefix=doc_prefix)
    index = faiss.IndexFlatIP(doc_emb.shape[1])
    index.add(doc_emb)

    qvec = encode_texts(model, [query_text], prefix=query_prefix)
    if alignment is not None:
        qvec = apply_procrustes_alignment(qvec, alignment)
        qvec = safe_l2_normalize(qvec)
    scores, indices = index.search(qvec, min(k, index.ntotal))

    gold_rank = None
    gold_score = None
    top_score = float(scores[0][0]) if len(scores[0]) else 0.0
    print("Query:", query_text)
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        row = context_pool.iloc[idx]
        preview = normalize_preview(row.doc_text[:200])
        is_gold = row.context_id == true_context_id
        marker = "*" if is_gold else "-"
        print(f"{marker} rank={rank} | score={float(score):.4f} | context_id={row.context_id} | {preview}")
        if is_gold:
            gold_rank = rank
            gold_score = float(score)
    explain_gold_score(gold_rank=gold_rank, gold_score=gold_score, top_score=top_score, k=k)

# Run MiniLM experiments
minilm = run_experiments_for_model("MiniLM")


In [ ]:
# MiniLM: query samples + top-k contexts
if "search_and_report" not in globals():
    raise RuntimeError("Run the helpers/experiments cell first.")
minilm = MODEL_RESULTS.get("MiniLM") or run_experiments_for_model("MiniLM")
alignment = minilm.get("alignment_retrieval")
model = minilm.get("model")
query_prefix = minilm.get("query_prefix", "")
doc_prefix = minilm.get("doc_prefix", "")

for qrow in test_queries.head(2).itertuples(index=False):
    context_pool = sample_context_pool(qrow.context_id, k=40, seed=42)
    print("Before alignment:")
    search_and_report(
        model=model,
        query_text=qrow.question,
        true_context_id=qrow.context_id,
        context_pool=context_pool,
        alignment=None,
        query_prefix=query_prefix,
        doc_prefix=doc_prefix,
        k=5,
    )

    print("After alignment:")
    search_and_report(
        model=model,
        query_text=qrow.question,
        true_context_id=qrow.context_id,
        context_pool=context_pool,
        alignment=alignment,
        query_prefix=query_prefix,
        doc_prefix=doc_prefix,
        k=5,
    )


In [ ]:
# BGE: experiments 1 + 2
bge = run_experiments_for_model("BGE")


In [ ]:
# BGE: query samples + top-k contexts
if "search_and_report" not in globals():
    raise RuntimeError("Run the helpers/experiments cell first.")
bge = MODEL_RESULTS.get("BGE") or run_experiments_for_model("BGE")
alignment = bge.get("alignment_retrieval")
model = bge.get("model")
query_prefix = bge.get("query_prefix", "")
doc_prefix = bge.get("doc_prefix", "")

for qrow in test_queries.head(2).itertuples(index=False):
    context_pool = sample_context_pool(qrow.context_id, k=40, seed=42)
    print("Before alignment:")
    search_and_report(
        model=model,
        query_text=qrow.question,
        true_context_id=qrow.context_id,
        context_pool=context_pool,
        alignment=None,
        query_prefix=query_prefix,
        doc_prefix=doc_prefix,
        k=5,
    )

    print("After alignment:")
    search_and_report(
        model=model,
        query_text=qrow.question,
        true_context_id=qrow.context_id,
        context_pool=context_pool,
        alignment=alignment,
        query_prefix=query_prefix,
        doc_prefix=doc_prefix,
        k=5,
    )


In [ ]:
# IntFloat (multilingual-e5-large): experiments 1 + 2
e5 = run_experiments_for_model("IntFloat")


In [ ]:
# IntFloat (multilingual-e5-large): query samples + top-k contexts
if "search_and_report" not in globals():
    raise RuntimeError("Run the helpers/experiments cell first.")
e5 = MODEL_RESULTS.get("IntFloat") or run_experiments_for_model("IntFloat")
alignment = e5.get("alignment_retrieval")
model = e5.get("model")
query_prefix = e5.get("query_prefix", "")
doc_prefix = e5.get("doc_prefix", "")

for qrow in test_queries.head(2).itertuples(index=False):
    context_pool = sample_context_pool(qrow.context_id, k=40, seed=42)
    print("Before alignment:")
    search_and_report(
        model=model,
        query_text=qrow.question,
        true_context_id=qrow.context_id,
        context_pool=context_pool,
        alignment=None,
        query_prefix=query_prefix,
        doc_prefix=doc_prefix,
        k=5,
    )

    print("After alignment:")
    search_and_report(
        model=model,
        query_text=qrow.question,
        true_context_id=qrow.context_id,
        context_pool=context_pool,
        alignment=alignment,
        query_prefix=query_prefix,
        doc_prefix=doc_prefix,
        k=5,
    )


## Extra open-source embeddings (experiments 1 + 2)


In [20]:
MODEL_CACHE: Dict[str, SentenceTransformer] = {}

def get_model(model_name: str) -> SentenceTransformer:
    if model_name not in MODEL_CACHE:
        MODEL_CACHE[model_name] = SentenceTransformer(model_name)
    return MODEL_CACHE[model_name]

def build_alignment_for_model(model: SentenceTransformer) -> Dict[str, Any]:
    if len(train_queries) == 0:
        raise ValueError("train_queries is empty; cannot learn alignment")
    ur_train_q = model.encode(
        train_queries["question"].tolist(),
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")
    doc_text_by_id = dict(zip(doc_df["context_id"], doc_df["doc_text"]))
    train_doc_texts = [doc_text_by_id[cid] for cid in train_queries["context_id"].tolist()]
    en_train_d = model.encode(
        train_doc_texts,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")
    return learn_procrustes_alignment(ur_train_q, en_train_d, center=True)

def explain_gold_score(*, gold_rank: Optional[int], gold_score: Optional[float], top_score: float, k: int) -> None:
    if gold_rank is None:
        print("Gold context not in the candidate pool (rank=N/A).")
        return
    if gold_rank == 1:
        print("Gold context is top-1; score is already the highest.")
        return
    gap = top_score - (gold_score or 0.0)
    print(
        f"Gold context rank={gold_rank} (top-{k}). Score gap vs top-1: {gap:.4f}. "
        "This usually means the Urdu query is closer to another English context than its paired gold context "
        "(translation mismatch, paraphrase drift, or imperfect alignment)."
    )

def search_and_report(
    *,
    model: SentenceTransformer,
    query_text: str,
    true_context_id: str,
    context_pool: pd.DataFrame,
    alignment: Optional[Dict[str, Any]],
    k: int = 5,
) -> None:
    index, _ = build_mini_index(context_pool, model=model)
    qvec = model.encode([query_text], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    if alignment is not None:
        qvec = apply_procrustes_alignment(qvec, alignment)
        qvec = safe_l2_normalize(qvec)
    scores, indices = index.search(qvec, min(k, index.ntotal))

    gold_rank = None
    gold_score = None
    top_score = float(scores[0][0]) if len(scores[0]) else 0.0
    print("Query:", query_text)
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        row = context_pool.iloc[idx]
        preview = row.doc_text[:200].replace("\n", " ")
        is_gold = row.context_id == true_context_id
        marker = "*" if is_gold else "-"
        print(f"{marker} rank={rank} | score={float(score):.4f} | context_id={row.context_id} | {preview}")
        if is_gold:
            gold_rank = rank
            gold_score = float(score)
    explain_gold_score(gold_rank=gold_rank, gold_score=gold_score, top_score=top_score, k=k)

def run_diagnostic(model_name: str, *, query_row: pd.Series, k: int = 5, pool_k: int = 40) -> None:
    print("\n=============================")
    print(f"Model: {model_name}")
    model = get_model(model_name)
    alignment = build_alignment_for_model(model)
    context_pool = sample_context_pool(query_row.context_id, k=pool_k, seed=42)

    print("\nBefore alignment:")
    search_and_report(
        model=model,
        query_text=query_row.question,
        true_context_id=query_row.context_id,
        context_pool=context_pool,
        alignment=None,
        k=k,
    )

    print("\nAfter alignment:")
    search_and_report(
        model=model,
        query_text=query_row.question,
        true_context_id=query_row.context_id,
        context_pool=context_pool,
        alignment=alignment,
        k=k,
    )

# Pick a test query and show chunks + gold-score diagnostics
example_query = test_queries.iloc[0]
run_diagnostic(EMBED_MODEL_NAME, query_row=example_query, k=5, pool_k=40)
run_diagnostic("BAAI/bge-m3", query_row=example_query, k=5, pool_k=40)


In [22]:
# Extra open-source embeddings (experiments 1 + 2)
if "compute_alignment_quality" not in globals():
    raise RuntimeError("Run the helpers/experiments cell first.")

OPEN_SOURCE_MODELS = [
    {"label": "LaBSE", "model_name": "sentence-transformers/LaBSE", "query_prefix": "", "doc_prefix": ""},
    {"label": "E5-Base", "model_name": "intfloat/multilingual-e5-base", "query_prefix": "query: ", "doc_prefix": "passage: "},
    {"label": "DistilUSE", "model_name": "sentence-transformers/distiluse-base-multilingual-cased-v2", "query_prefix": "", "doc_prefix": ""},
]

def run_custom_experiments(label: str, model_name: str, query_prefix: str, doc_prefix: str) -> None:
    model = get_model(model_name)
    print(f"=== {label} ({model_name}) ===")
    compute_alignment_quality(model, label=label, query_prefix=query_prefix)
    compute_retrieval_metrics(
        model,
        label=label,
        query_prefix=query_prefix,
        doc_prefix=doc_prefix,
    )

for cfg in OPEN_SOURCE_MODELS:
    run_custom_experiments(
        cfg["label"],
        cfg["model_name"],
        cfg["query_prefix"],
        cfg["doc_prefix"],
    )



In [ ]:
run_diagnostic("sentence-transformers/LaBSE", query_row=example_query, k=5, pool_k=40)
run_diagnostic("intfloat/multilingual-e5-base", query_row=example_query, k=5, pool_k=40)
run_diagnostic("sentence-transformers/distiluse-base-multilingual-cased-v2", query_row=example_query, k=5, pool_k=40)


## Filtered context-pair experiments by cosine similarity threshold

In [ ]:
# Filtered context-pair experiments by cosine similarity threshold
if "encode_texts" not in globals():
    raise RuntimeError("Run the helpers/experiments cell first.")

FILTER_THRESHOLD = 0.85  # adjust if you want stricter/looser filtering
FILTER_MODELS = [
    {"label": "MiniLM", "model_name": EMBED_MODEL_NAME, "query_prefix": "", "doc_prefix": ""},
    {"label": "BGE", "model_name": "BAAI/bge-m3", "query_prefix": "", "doc_prefix": ""},
    {"label": "IntFloat", "model_name": "intfloat/multilingual-e5-large", "query_prefix": "query: ", "doc_prefix": "passage: "},
]

def build_filtered_dataset(model: SentenceTransformer, *, doc_prefix: str, threshold: float) -> Dict[str, Any]:
    en_ctx = encode_texts(model, context_pairs["context_squad"].tolist(), prefix=doc_prefix)
    ur_ctx = encode_texts(model, context_pairs["context_uqa"].tolist(), prefix=doc_prefix)
    cos_sim = np.sum(en_ctx * ur_ctx, axis=1)
    keep_mask = cos_sim >= threshold
    filtered_pairs = context_pairs[keep_mask].reset_index(drop=True).copy()
    filtered_pairs["context_id"] = [f"fctx_{i}" for i in range(len(filtered_pairs))]

    context_id_en = dict(zip(filtered_pairs["context_squad"], filtered_pairs["context_id"]))
    context_id_ur = dict(zip(filtered_pairs["context_uqa"], filtered_pairs["context_id"]))

    df_filtered = df.copy()
    df_filtered["context_id"] = df_filtered.apply(
        lambda r: context_id_en.get(r.context) if r.language == "en" else context_id_ur.get(r.context),
        axis=1,
    )
    df_filtered = df_filtered.dropna(subset=["context_id"]).reset_index(drop=True)

    query_df_filtered = (
        df_filtered[df_filtered["language"] == "ur"]
        .drop_duplicates(subset=["context_id"])
        .loc[:, ["context_id", "question"]]
        .reset_index(drop=True)
    )

    question_pairs_filtered = (
        df_filtered.pivot_table(index="pair_id", columns="language", values="question", aggfunc="first")
        .dropna(subset=["en", "ur"])
        .reset_index()
    )

    return {
        "context_pairs": filtered_pairs,
        "df": df_filtered,
        "query_df": query_df_filtered,
        "question_pairs": question_pairs_filtered,
        "cos_sim": cos_sim,
    }

def run_filtered_experiments(cfg: Dict[str, str], threshold: float) -> None:
    model = get_model(cfg["model_name"])
    label = f"{cfg['label']} (filtered>= {threshold})"
    data = build_filtered_dataset(model, doc_prefix=cfg["doc_prefix"], threshold=threshold)

    print(f"=== {label} ===")
    print("Filtered context pairs:", len(data["context_pairs"]))

    # Experiment 1 (alignment quality on questions)
    qp = data["question_pairs"].sample(frac=1.0, random_state=42).reset_index(drop=True)
    cut = int(len(qp) * ALIGN_TRAIN_RATIO)
    train_q_f = qp.iloc[:cut]
    test_q_f = qp.iloc[cut:]

    en_train = encode_texts(model, train_q_f["en"].tolist(), prefix=cfg["query_prefix"])
    ur_train = encode_texts(model, train_q_f["ur"].tolist(), prefix=cfg["query_prefix"])
    ur_train, en_train = filter_empty_vectors(ur_train, en_train)
    if len(ur_train) == 0:
        print("No valid train vectors after filtering.")
        return
    alignment_q = learn_procrustes_alignment(ur_train, en_train, center=True)

    en_test = encode_texts(model, test_q_f["en"].tolist(), prefix=cfg["query_prefix"])
    ur_test = encode_texts(model, test_q_f["ur"].tolist(), prefix=cfg["query_prefix"])
    ur_test, en_test = filter_empty_vectors(ur_test, en_test)
    dist_before = cosine_distance(ur_test, en_test).mean()
    dist_after = cosine_distance(apply_procrustes_alignment(ur_test, alignment_q), en_test).mean()

    rand_n = min(ALIGN_RANDOM_BASELINE_N, len(en_test))
    rand_idx = np.random.choice(len(en_test), size=rand_n, replace=False)
    rand_en = en_test[rand_idx]
    rand_ur = ur_test[rand_idx]
    np.random.shuffle(rand_en)
    dist_random = cosine_distance(rand_ur, rand_en).mean()

    print("Experiment 1 (cosine distance):", {
        "random": round(float(dist_random), 4),
        "original": round(float(dist_before), 4),
        "aligned": round(float(dist_after), 4),
    })

    # Experiment 2 (retrieval)
    query_pool = data["query_df"].sample(frac=1.0, random_state=42).reset_index(drop=True)
    cut = int(len(query_pool) * RETRIEVAL_TRAIN_RATIO)
    train_queries_f = query_pool.iloc[:cut].reset_index(drop=True)
    test_queries_f = query_pool.iloc[cut:].reset_index(drop=True)

    doc_df_f = data["context_pairs"][["context_id", "context_squad"]].rename(columns={"context_squad": "doc_text"})
    doc_text_by_id = dict(zip(doc_df_f["context_id"], doc_df_f["doc_text"]))

    ur_train_q = encode_texts(model, train_queries_f["question"].tolist(), prefix=cfg["query_prefix"])
    train_doc_texts = [doc_text_by_id[cid] for cid in train_queries_f["context_id"].tolist()]
    en_train_d = encode_texts(model, train_doc_texts, prefix=cfg["doc_prefix"])
    ur_train_q, en_train_d = filter_empty_vectors(ur_train_q, en_train_d)
    if len(ur_train_q) == 0:
        print("No valid train vectors after filtering.")
        return
    alignment_retrieval = learn_procrustes_alignment(ur_train_q, en_train_d, center=True)

    test_doc_mask = doc_df_f["context_id"].isin(test_queries_f["context_id"]).to_numpy()
    test_doc_df = doc_df_f[test_doc_mask].reset_index(drop=True)
    test_doc_embeddings = encode_texts(model, test_doc_df["doc_text"].tolist(), prefix=cfg["doc_prefix"])
    test_index = faiss.IndexFlatIP(test_doc_embeddings.shape[1])
    test_index.add(test_doc_embeddings)

    def rank_query(query_text: str, true_context_id: str, alignment: Optional[Dict[str, Any]] = None) -> Optional[int]:
        qvec = encode_texts(model, [query_text], prefix=cfg["query_prefix"])
        if alignment is not None:
            qvec = apply_procrustes_alignment(qvec, alignment)
            qvec = safe_l2_normalize(qvec)
        scores, indices = test_index.search(qvec, test_index.ntotal)
        rank = 1
        for idx in indices[0]:
            if idx < 0:
                break
            if test_doc_df.iloc[idx]["context_id"] == true_context_id:
                return rank
            rank += 1
        return None

    ranks_before = []
    ranks_after = []
    for row in tqdm(test_queries_f.itertuples(index=False), total=len(test_queries_f), desc=f"{label} retrieval eval"):
        ranks_before.append(rank_query(row.question, row.context_id, alignment=None))
        ranks_after.append(rank_query(row.question, row.context_id, alignment=alignment_retrieval))

    print("Experiment 2 (recall):")
    print("  Before alignment:", recall_at_k(ranks_before, RETRIEVAL_K_VALUES))
    print("  After alignment:", recall_at_k(ranks_after, RETRIEVAL_K_VALUES))

for cfg in FILTER_MODELS:
    run_filtered_experiments(cfg, FILTER_THRESHOLD)


In [ ]:
# Filter by question-context cosine similarity, then run retrieval
if "encode_texts" not in globals():
    raise RuntimeError("Run the helpers/experiments cell first.")

QCOS_THRESHOLD = 0.75  # increase to be stricter
QCOS_MODEL = {
    "label": "IntFloat",
    "model_name": "intfloat/multilingual-e5-large",
    "query_prefix": "query: ",
    "doc_prefix": "passage: ",
}

def run_qc_filtered_retrieval(cfg: Dict[str, str], threshold: float) -> None:
    model = get_model(cfg["model_name"])
    label = f"{cfg['label']} (q-ctx >= {threshold})"

    # Build English doc table
    doc_df_local = context_pairs[["context_id", "context_squad"]].rename(columns={"context_squad": "doc_text"})
    doc_text_by_id = dict(zip(doc_df_local["context_id"], doc_df_local["doc_text"]))

    # Compute per-query similarity to its gold context
    q_texts = query_df["question"].tolist()
    ctx_texts = [doc_text_by_id[cid] for cid in query_df["context_id"].tolist()]
    q_emb = encode_texts(model, q_texts, prefix=cfg["query_prefix"])
    ctx_emb = encode_texts(model, ctx_texts, prefix=cfg["doc_prefix"])
    q_emb, ctx_emb = filter_empty_vectors(q_emb, ctx_emb)

    cos_sim = np.sum(q_emb * ctx_emb, axis=1)
    keep_mask = cos_sim >= threshold

    filtered_queries = query_df.iloc[: len(keep_mask)].copy()
    filtered_queries = filtered_queries.loc[keep_mask].reset_index(drop=True)
    print(f"{label}: kept {len(filtered_queries):,} / {len(query_df):,} queries")

    # Split and run retrieval
    query_pool = filtered_queries.sample(frac=1.0, random_state=42).reset_index(drop=True)
    cut = int(len(query_pool) * RETRIEVAL_TRAIN_RATIO)
    train_queries_f = query_pool.iloc[:cut].reset_index(drop=True)
    test_queries_f = query_pool.iloc[cut:].reset_index(drop=True)

    ur_train_q = encode_texts(model, train_queries_f["question"].tolist(), prefix=cfg["query_prefix"])
    train_doc_texts = [doc_text_by_id[cid] for cid in train_queries_f["context_id"].tolist()]
    en_train_d = encode_texts(model, train_doc_texts, prefix=cfg["doc_prefix"])
    ur_train_q, en_train_d = filter_empty_vectors(ur_train_q, en_train_d)
    if len(ur_train_q) == 0:
        print("No valid train vectors after filtering.")
        return
    alignment_retrieval = learn_procrustes_alignment(ur_train_q, en_train_d, center=True)

    test_doc_mask = doc_df_local["context_id"].isin(test_queries_f["context_id"]).to_numpy()
    test_doc_df = doc_df_local[test_doc_mask].reset_index(drop=True)
    test_doc_embeddings = encode_texts(model, test_doc_df["doc_text"].tolist(), prefix=cfg["doc_prefix"])
    test_index = faiss.IndexFlatIP(test_doc_embeddings.shape[1])
    test_index.add(test_doc_embeddings)

    def rank_query(query_text: str, true_context_id: str, alignment: Optional[Dict[str, Any]] = None) -> Optional[int]:
        qvec = encode_texts(model, [query_text], prefix=cfg["query_prefix"])
        if alignment is not None:
            qvec = apply_procrustes_alignment(qvec, alignment)
            qvec = safe_l2_normalize(qvec)
        scores, indices = test_index.search(qvec, test_index.ntotal)
        rank = 1
        for idx in indices[0]:
            if idx < 0:
                break
            if test_doc_df.iloc[idx]["context_id"] == true_context_id:
                return rank
            rank += 1
        return None

    ranks_before = []
    ranks_after = []
    for row in tqdm(test_queries_f.itertuples(index=False), total=len(test_queries_f), desc=f"{label} retrieval eval"):
        ranks_before.append(rank_query(row.question, row.context_id, alignment=None))
        ranks_after.append(rank_query(row.question, row.context_id, alignment=alignment_retrieval))

    print("Retrieval (filtered by q-ctx similarity):")
    print("  Before alignment:", recall_at_k(ranks_before, RETRIEVAL_K_VALUES))
    print("  After alignment:", recall_at_k(ranks_after, RETRIEVAL_K_VALUES))

run_qc_filtered_retrieval(QCOS_MODEL, QCOS_THRESHOLD)


In [ ]:

run_qc_filtered_retrieval(QCOS_MODEL, 0.78)


In [ ]:
run_qc_filtered_retrieval(QCOS_MODEL, 0.9)


In [ ]:
# RAGAS end-to-end eval for LaBSE (before/after alignment)
# Requires: ragas, datasets, and an LLM key for faithful metrics.

!pip install -q ragas datasets

if "encode_texts" not in globals():
    raise RuntimeError("Run the helpers/experiments cell first.")

try:
    from ragas import evaluate
    from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
    from datasets import Dataset
except Exception as exc:
    raise RuntimeError("Install ragas + datasets to run this cell.") from exc

RAGAS_SAMPLE_N = 100
RAGAS_TOP_K = 3

LABSE_NAME = "sentence-transformers/LaBSE"
labse = get_model(LABSE_NAME)

# Ensure merged is available for answers
if "merged" not in globals():
    merged = pd.read_csv("Squad.csv")

# Build LaBSE doc embeddings for retrieval
labse_doc_df = context_pairs[["context_id", "context_squad"]].rename(columns={"context_squad": "doc_text"})
labse_doc_emb = encode_texts(labse, labse_doc_df["doc_text"].tolist())
labse_index = faiss.IndexFlatIP(labse_doc_emb.shape[1])
labse_index.add(labse_doc_emb)

# Learn query->doc alignment for LaBSE
ur_train_q = encode_texts(labse, train_queries["question"].tolist())
doc_text_by_id = dict(zip(labse_doc_df["context_id"], labse_doc_df["doc_text"]))
train_doc_texts = [doc_text_by_id[cid] for cid in train_queries["context_id"].tolist()]
en_train_d = encode_texts(labse, train_doc_texts)

ur_train_q, en_train_d = filter_empty_vectors(ur_train_q, en_train_d)
labse_alignment = learn_procrustes_alignment(ur_train_q, en_train_d, center=True)


def pick_answer(row: pd.Series) -> str:
    ans = row.get("answers")
    if isinstance(ans, str):
        try:
            import json as _json
            ans_obj = _json.loads(ans)
            if isinstance(ans_obj, dict) and ans_obj.get("text"):
                return str(ans_obj["text"][0])
        except Exception:
            pass
    alt = row.get("answer")
    return str(alt) if isinstance(alt, str) and alt.strip() else ""


def naive_answer(contexts: List[str]) -> str:
    if not contexts:
        return ""
    first = contexts[0].strip()
    return first.split(".", 1)[0].strip() + "."


def retrieve_contexts(query: str, *, alignment: Optional[Dict[str, Any]] = None, k: int = 3) -> List[str]:
    qvec = encode_texts(labse, [query])
    if alignment is not None:
        qvec = apply_procrustes_alignment(qvec, alignment)
        qvec = safe_l2_normalize(qvec)
    scores, indices = labse_index.search(qvec, min(k, labse_index.ntotal))
    ctxs = []
    for idx in indices[0]:
        if idx < 0:
            continue
        ctxs.append(labse_doc_df.iloc[idx]["doc_text"])
    return ctxs


def build_ragas_dataset(*, alignment: Optional[Dict[str, Any]] = None) -> Dataset:
    sample = merged.dropna(subset=["question_uqa", "context_squad"]).sample(
        n=min(RAGAS_SAMPLE_N, len(merged)), random_state=42
    )
    questions = []
    answers = []
    contexts = []
    ground_truths = []

    for row in sample.itertuples(index=False):
        row = pd.Series(row._asdict())
        q = row.get("question_uqa")
        if not isinstance(q, str) or not q.strip():
            continue
        ctxs = retrieve_contexts(q, alignment=alignment, k=RAGAS_TOP_K)
        gt = pick_answer(row)
        questions.append(q)
        contexts.append(ctxs)
        answers.append(naive_answer(ctxs))
        ground_truths.append(gt)

    data = {
        "question": questions,
        "answer": answers,
        "contexts": contexts,
        "ground_truth": ground_truths,
    }
    return Dataset.from_dict(data)


# Before alignment
ragas_before = build_ragas_dataset(alignment=None)
results_before = evaluate(
    ragas_before,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
)
print("LaBSE RAGAS (before alignment):")
print(results_before)

# After alignment
ragas_after = build_ragas_dataset(alignment=labse_alignment)
results_after = evaluate(
    ragas_after,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
)
print("LaBSE RAGAS (after alignment):")
print(results_after)


In [23]:
# RAGAS end-to-end eval for LaBSE (before/after alignment)
# Requires: ragas, datasets, langchain-openai, langchain-community, and an LLM key for faithful metrics.

!pip install -q ragas datasets langchain-openai langchain-community

if "encode_texts" not in globals():
    raise RuntimeError("Run the helpers/experiments cell first.")

try:
    from ragas import evaluate
    from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
    from datasets import Dataset
    from langchain_openai import ChatOpenAI
    from langchain_community.embeddings import HuggingFaceEmbeddings
except Exception as exc:
    raise RuntimeError("Install ragas + datasets + langchain-openai + langchain-community to run this cell.") from exc

RAGAS_SAMPLE_N = 100
RAGAS_TOP_K = 3

LABSE_NAME = "sentence-transformers/LaBSE"
labse = get_model(LABSE_NAME)

# LLM for ragas (set n=3 to satisfy multi-generation expectation; temperature=0 for determinism)
ragas_llm = ChatOpenAI(model=os.getenv("RAGAS_LLM_MODEL", "gpt-4.1-mini"), temperature=0, max_tokens=512, n=3)
# Embeddings for ragas (HuggingFace backend exposes embed_query/embed_documents)
ragas_embeddings = HuggingFaceEmbeddings(model_name=LABSE_NAME, encode_kwargs={"normalize_embeddings": True})

# Ensure merged is available for answers
if "merged" not in globals():
    merged = pd.read_csv("Squad.csv")

# Build LaBSE doc embeddings for retrieval
labse_doc_df = context_pairs[["context_id", "context_squad"]].rename(columns={"context_squad": "doc_text"})
labse_doc_emb = encode_texts(labse, labse_doc_df["doc_text"].tolist())
labse_index = faiss.IndexFlatIP(labse_doc_emb.shape[1])
labse_index.add(labse_doc_emb)

# Learn query->doc alignment for LaBSE
ur_train_q = encode_texts(labse, train_queries["question"].tolist())
doc_text_by_id = dict(zip(labse_doc_df["context_id"], labse_doc_df["doc_text"]))
train_doc_texts = [doc_text_by_id[cid] for cid in train_queries["context_id"].tolist()]
en_train_d = encode_texts(labse, train_doc_texts)

ur_train_q, en_train_d = filter_empty_vectors(ur_train_q, en_train_d)
labse_alignment = learn_procrustes_alignment(ur_train_q, en_train_d, center=True)


def pick_answer(row: pd.Series) -> str:
    ans = row.get("answers")
    if isinstance(ans, str):
        try:
            import json as _json
            ans_obj = _json.loads(ans)
            if isinstance(ans_obj, dict) and ans_obj.get("text"):
                return str(ans_obj["text"][0])
        except Exception:
            pass
    alt = row.get("answer")
    return str(alt) if isinstance(alt, str) and alt.strip() else ""


def naive_answer(contexts: List[str]) -> str:
    if not contexts:
        return ""
    first = contexts[0].strip()
    return first.split(".", 1)[0].strip() + "."


def retrieve_contexts(query: str, *, alignment: Optional[Dict[str, Any]] = None, k: int = 3) -> List[str]:
    qvec = encode_texts(labse, [query])
    if alignment is not None:
        qvec = apply_procrustes_alignment(qvec, alignment)
        qvec = safe_l2_normalize(qvec)
    scores, indices = labse_index.search(qvec, min(k, labse_index.ntotal))
    ctxs = []
    for idx in indices[0]:
        if idx < 0:
            continue
        ctxs.append(labse_doc_df.iloc[idx]["doc_text"])
    return ctxs


def build_ragas_dataset(*, alignment: Optional[Dict[str, Any]] = None) -> Dataset:
    sample = merged.dropna(subset=["question_uqa", "context_squad"]).sample(
        n=min(RAGAS_SAMPLE_N, len(merged)), random_state=42
    )
    questions = []
    answers = []
    contexts = []
    ground_truths = []

    for row in sample.itertuples(index=False):
        row = pd.Series(row._asdict())
        q = row.get("question_uqa")
        if not isinstance(q, str) or not q.strip():
            continue
        ctxs = retrieve_contexts(q, alignment=alignment, k=RAGAS_TOP_K)
        gt = pick_answer(row)
        questions.append(q)
        contexts.append(ctxs)
        answers.append(naive_answer(ctxs))
        ground_truths.append(gt)

    data = {
        "question": questions,
        "answer": answers,
        "contexts": contexts,
        "ground_truth": ground_truths,
    }
    return Dataset.from_dict(data)


# Before alignment
ragas_before = build_ragas_dataset(alignment=None)
results_before = evaluate(
    ragas_before,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)
print("LaBSE RAGAS (before alignment):")
print(results_before)

# After alignment
ragas_after = build_ragas_dataset(alignment=labse_alignment)
results_after = evaluate(
    ragas_after,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)
print("LaBSE RAGAS (after alignment):")
print(results_after)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 kB 3.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.7/84.7 kB 1.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 5.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 5.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 489.1/489.1 kB 5.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 4.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 3.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 6.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 2.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 9.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.1/1

RuntimeError: Install ragas + datasets + langchain-openai + langchain-community to run this cell.

In [26]:
# RAGAS end-to-end eval for LaBSE (before/after alignment)
# Requires: ragas, datasets, langchain-openai, langchain-community, and an LLM key for faithful metrics.

!pip install -q ragas datasets langchain-openai langchain-community

import os

# Avoid tokenizers parallelism warning when forking
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

if "encode_texts" not in globals():
    raise RuntimeError("Run the helpers/experiments cell first.")

try:
    from ragas import evaluate
    from datasets import Dataset
    from langchain_openai import ChatOpenAI, OpenAIEmbeddings
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except Exception:
        from langchain_community.embeddings import HuggingFaceEmbeddings
except Exception as exc:
    raise RuntimeError("Install ragas + datasets + langchain-openai + langchain-community to run this cell.") from exc

RAGAS_SAMPLE_N = 100
RAGAS_TOP_K = 3

LABSE_NAME = "sentence-transformers/LaBSE"
labse = get_model(LABSE_NAME)

# LLM for ragas (n=1 to avoid multi-generation warnings; temperature=0 for determinism)
ragas_llm = ChatOpenAI(model=os.getenv("RAGAS_LLM_MODEL", "gpt-4.1-mini"), temperature=0, max_tokens=512, n=1)

# RAGAS embeddings to compare (add top-accuracy models)
RAGAS_EMBED_MODELS = [
    {"label": "LaBSE", "model_name": "sentence-transformers/LaBSE"},
    {"label": "IntFloat", "model_name": "intfloat/multilingual-e5-large"},
    {"label": "BGE", "model_name": "BAAI/bge-m3"},
]

# Resolve RAGAS metrics across versions
try:
    from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
    ragas_metrics = [Faithfulness(), AnswerRelevancy(), ContextPrecision(), ContextRecall()]
except Exception:
    from ragas.metrics.collections import faithfulness, answer_relevancy, context_precision, context_recall
    try:
        from ragas.metrics.base import Metric
    except Exception:
        Metric = None

    def to_metric(m):
        if Metric is not None and isinstance(m, Metric):
            return m
        if callable(m):
            return m()
        return m

    ragas_metrics = [
        to_metric(faithfulness),
        to_metric(answer_relevancy),
        to_metric(context_precision),
        to_metric(context_recall),
    ]

    if Metric is not None and any(not isinstance(m, Metric) for m in ragas_metrics):
        raise RuntimeError("RAGAS metrics failed to initialize as Metric objects. Check ragas version/imports.")

# Ensure merged is available for answers
if "merged" not in globals():
    merged = pd.read_csv("Squad.csv")

# Build LaBSE doc embeddings for retrieval
labse_doc_df = context_pairs[["context_id", "context_squad"]].rename(columns={"context_squad": "doc_text"})
labse_doc_emb = encode_texts(labse, labse_doc_df["doc_text"].tolist())
labse_index = faiss.IndexFlatIP(labse_doc_emb.shape[1])
labse_index.add(labse_doc_emb)

# Learn query->doc alignment for LaBSE
ur_train_q = encode_texts(labse, train_queries["question"].tolist())
doc_text_by_id = dict(zip(labse_doc_df["context_id"], labse_doc_df["doc_text"]))
train_doc_texts = [doc_text_by_id[cid] for cid in train_queries["context_id"].tolist()]
en_train_d = encode_texts(labse, train_doc_texts)

ur_train_q, en_train_d = filter_empty_vectors(ur_train_q, en_train_d)
labse_alignment = learn_procrustes_alignment(ur_train_q, en_train_d, center=True)


def pick_answer(row: pd.Series) -> str:
    ans = row.get("answers")
    if isinstance(ans, str):
        try:
            import json as _json
            ans_obj = _json.loads(ans)
            if isinstance(ans_obj, dict) and ans_obj.get("text"):
                return str(ans_obj["text"][0])
        except Exception:
            pass
    alt = row.get("answer")
    return str(alt) if isinstance(alt, str) and alt.strip() else ""


def naive_answer(contexts: List[str]) -> str:
    if not contexts:
        return ""
    first = contexts[0].strip()
    return first.split(".", 1)[0].strip() + "."


def retrieve_contexts(query: str, *, alignment: Optional[Dict[str, Any]] = None, k: int = 3) -> List[str]:
    qvec = encode_texts(labse, [query])
    if alignment is not None:
        qvec = apply_procrustes_alignment(qvec, alignment)
        qvec = safe_l2_normalize(qvec)
    scores, indices = labse_index.search(qvec, min(k, labse_index.ntotal))
    ctxs = []
    for idx in indices[0]:
        if idx < 0:
            continue
        ctxs.append(labse_doc_df.iloc[idx]["doc_text"])
    return ctxs


def build_ragas_dataset(*, alignment: Optional[Dict[str, Any]] = None) -> Dataset:
    sample = merged.dropna(subset=["question_uqa", "context_squad"]).sample(
        n=min(RAGAS_SAMPLE_N, len(merged)), random_state=42
    )
    questions = []
    answers = []
    contexts = []
    ground_truths = []

    for row in sample.itertuples(index=False):
        row = pd.Series(row._asdict())
        q = row.get("question_uqa")
        if not isinstance(q, str) or not q.strip():
            continue
        ctxs = retrieve_contexts(q, alignment=alignment, k=RAGAS_TOP_K)
        gt = pick_answer(row)
        ans = naive_answer(ctxs)
        if not ans:
            continue
        questions.append(q)
        contexts.append(ctxs)
        answers.append(ans)
        ground_truths.append(gt)

    data = {
        "question": questions,
        "answer": answers,
        "contexts": contexts,
        "ground_truth": ground_truths,
    }
    return Dataset.from_dict(data)


def build_ragas_embeddings(model_name: str):
    emb = HuggingFaceEmbeddings(model_name=model_name, encode_kwargs={"normalize_embeddings": True})
    if not hasattr(emb, "embed_query"):
        emb = OpenAIEmbeddings(model=os.getenv("RAGAS_EMBEDDING_MODEL", "text-embedding-3-small"))
    return emb


# Build datasets once (retrieval uses LaBSE)
ragas_before = build_ragas_dataset(alignment=None)
ragas_after = build_ragas_dataset(alignment=labse_alignment)

for cfg in RAGAS_EMBED_MODELS:
    ragas_embeddings = build_ragas_embeddings(cfg["model_name"])

    results_before = evaluate(
        ragas_before,
        metrics=ragas_metrics,
        llm=ragas_llm,
        embeddings=ragas_embeddings,
    )
    print(f"{cfg['label']} RAGAS (before alignment):")
    print(results_before)

    results_after = evaluate(
        ragas_after,
        metrics=ragas_metrics,
        llm=ragas_llm,
        embeddings=ragas_embeddings,
    )
    print(f"{cfg['label']} RAGAS (after alignment):")
    print(results_after)


/tmp/ipykernel_55/2925096878.py:43: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
/tmp/ipykernel_55/2925096878.py:43: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
/tmp/ipykernel_55/2925096878.py:43: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import Faithfulness, AnswerRel

Evaluating:   0%|          | 0/400 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[48]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)
ERROR:ragas.executor:Exception raised in Job[104]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)
ERROR:ragas.executor:Exception raised in Job[236]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)


LaBSE RAGAS (before alignment):
{'faithfulness': 0.9254, 'answer_relevancy': 0.4438, 'context_precision': 0.2675, 'context_recall': 0.2900}


Evaluating:   0%|          | 0/400 [00:00<?, ?it/s]

...

ERROR:ragas.executor:Exception raised in Job[48]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)
ERROR:ragas.executor:Exception raised in Job[104]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)
ERROR:ragas.executor:Exception raised in Job[236]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)


BGE RAGAS (before alignment):
{'faithfulness': 0.9246, 'answer_relevancy': 0.3973, 'context_precision': 0.2542, 'context_recall': 0.3100}


Evaluating:   0%|          | 0/400 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[236]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)
ERROR:ragas.executor:Exception raised in Job[356]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)


BGE RAGAS (after alignment):
{'faithfulness': 0.9435, 'answer_relevancy': 0.4595, 'context_precision': 0.3758, 'context_recall': 0.4100}


In [27]:
# RAGAS end-to-end eval for LaBSE (before/after alignment)
# Requires: ragas, datasets, langchain-openai, langchain-community, and an LLM key for faithful metrics.

!pip install -q ragas datasets langchain-openai langchain-community

import os

# Avoid tokenizers parallelism warning when forking
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

if "encode_texts" not in globals():
    raise RuntimeError("Run the helpers/experiments cell first.")

try:
    from ragas import evaluate
    from datasets import Dataset
    from langchain_openai import ChatOpenAI, OpenAIEmbeddings
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except Exception:
        from langchain_community.embeddings import HuggingFaceEmbeddings
except Exception as exc:
    raise RuntimeError("Install ragas + datasets + langchain-openai + langchain-community to run this cell.") from exc

RAGAS_SAMPLE_N = 100
RAGAS_TOP_K = 3

LABSE_NAME = "sentence-transformers/LaBSE"
labse = get_model(LABSE_NAME)

# LLM for ragas (n=1 to avoid multi-generation warnings; temperature=0 for determinism)
ragas_llm = ChatOpenAI(model=os.getenv("RAGAS_LLM_MODEL", "gpt-4.1-mini"), temperature=0, max_tokens=512, n=1)

# RAGAS embeddings to compare (add top-accuracy models)
RAGAS_EMBED_MODELS = [
    {"label": "MiniLM", "model_name": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"},
    {"label": "BGE", "model_name": "BAAI/bge-m3"},
    {"label": "IntFloat", "model_name": "intfloat/multilingual-e5-large"},
    {"label": "LaBSE", "model_name": "sentence-transformers/LaBSE"},
    {"label": "E5-Base", "model_name": "intfloat/multilingual-e5-base"},
    {"label": "DistilUSE", "model_name": "sentence-transformers/distiluse-base-multilingual-cased-v2"},
]

# Resolve RAGAS metrics across versions
try:
    from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
    ragas_metrics = [Faithfulness(), AnswerRelevancy(), ContextPrecision(), ContextRecall()]
except Exception:
    from ragas.metrics.collections import faithfulness, answer_relevancy, context_precision, context_recall
    try:
        from ragas.metrics.base import Metric
    except Exception:
        Metric = None

    def to_metric(m):
        if Metric is not None and isinstance(m, Metric):
            return m
        if callable(m):
            return m()
        return m

    ragas_metrics = [
        to_metric(faithfulness),
        to_metric(answer_relevancy),
        to_metric(context_precision),
        to_metric(context_recall),
    ]

    if Metric is not None and any(not isinstance(m, Metric) for m in ragas_metrics):
        raise RuntimeError("RAGAS metrics failed to initialize as Metric objects. Check ragas version/imports.")

# Ensure merged is available for answers
if "merged" not in globals():
    merged = pd.read_csv("Squad.csv")

# Build LaBSE doc embeddings for retrieval
labse_doc_df = context_pairs[["context_id", "context_squad"]].rename(columns={"context_squad": "doc_text"})
labse_doc_emb = encode_texts(labse, labse_doc_df["doc_text"].tolist())
labse_index = faiss.IndexFlatIP(labse_doc_emb.shape[1])
labse_index.add(labse_doc_emb)

# Learn query->doc alignment for LaBSE
ur_train_q = encode_texts(labse, train_queries["question"].tolist())
doc_text_by_id = dict(zip(labse_doc_df["context_id"], labse_doc_df["doc_text"]))
train_doc_texts = [doc_text_by_id[cid] for cid in train_queries["context_id"].tolist()]
en_train_d = encode_texts(labse, train_doc_texts)

ur_train_q, en_train_d = filter_empty_vectors(ur_train_q, en_train_d)
labse_alignment = learn_procrustes_alignment(ur_train_q, en_train_d, center=True)


def pick_answer(row: pd.Series) -> str:
    ans = row.get("answers")
    if isinstance(ans, str):
        try:
            import json as _json
            ans_obj = _json.loads(ans)
            if isinstance(ans_obj, dict) and ans_obj.get("text"):
                return str(ans_obj["text"][0])
        except Exception:
            pass
    alt = row.get("answer")
    return str(alt) if isinstance(alt, str) and alt.strip() else ""


def naive_answer(contexts: List[str]) -> str:
    if not contexts:
        return ""
    first = contexts[0].strip()
    return first.split(".", 1)[0].strip() + "."


def retrieve_contexts(query: str, *, alignment: Optional[Dict[str, Any]] = None, k: int = 3) -> List[str]:
    qvec = encode_texts(labse, [query])
    if alignment is not None:
        qvec = apply_procrustes_alignment(qvec, alignment)
        qvec = safe_l2_normalize(qvec)
    scores, indices = labse_index.search(qvec, min(k, labse_index.ntotal))
    ctxs = []
    for idx in indices[0]:
        if idx < 0:
            continue
        ctxs.append(labse_doc_df.iloc[idx]["doc_text"])
    return ctxs


def build_ragas_dataset(*, alignment: Optional[Dict[str, Any]] = None) -> Dataset:
    sample = merged.dropna(subset=["question_uqa", "context_squad"]).sample(
        n=min(RAGAS_SAMPLE_N, len(merged)), random_state=42
    )
    questions = []
    answers = []
    contexts = []
    ground_truths = []

    for row in sample.itertuples(index=False):
        row = pd.Series(row._asdict())
        q = row.get("question_uqa")
        if not isinstance(q, str) or not q.strip():
            continue
        ctxs = retrieve_contexts(q, alignment=alignment, k=RAGAS_TOP_K)
        gt = pick_answer(row)
        ans = naive_answer(ctxs)
        if not ans:
            continue
        questions.append(q)
        contexts.append(ctxs)
        answers.append(ans)
        ground_truths.append(gt)

    data = {
        "question": questions,
        "answer": answers,
        "contexts": contexts,
        "ground_truth": ground_truths,
    }
    return Dataset.from_dict(data)


def build_ragas_embeddings(model_name: str):
    emb = HuggingFaceEmbeddings(model_name=model_name, encode_kwargs={"normalize_embeddings": True})
    if not hasattr(emb, "embed_query"):
        emb = OpenAIEmbeddings(model=os.getenv("RAGAS_EMBEDDING_MODEL", "text-embedding-3-small"))
    return emb


# Build datasets once (retrieval uses LaBSE)
ragas_before = build_ragas_dataset(alignment=None)
ragas_after = build_ragas_dataset(alignment=labse_alignment)

for cfg in RAGAS_EMBED_MODELS:
    ragas_embeddings = build_ragas_embeddings(cfg["model_name"])

    results_before = evaluate(
        ragas_before,
        metrics=ragas_metrics,
        llm=ragas_llm,
        embeddings=ragas_embeddings,
    )
    print(f"{cfg['label']} RAGAS (before alignment):")
    print(results_before)

    results_after = evaluate(
        ragas_after,
        metrics=ragas_metrics,
        llm=ragas_llm,
        embeddings=ragas_embeddings,
    )
    print(f"{cfg['label']} RAGAS (after alignment):")
    print(results_after)


/tmp/ipykernel_55/2547124639.py:46: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
/tmp/ipykernel_55/2547124639.py:46: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
/tmp/ipykernel_55/2547124639.py:46: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import Faithfulness, AnswerRel

Evaluating:   0%|          | 0/400 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[48]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)
ERROR:ragas.executor:Exception raised in Job[104]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)
ERROR:ragas.executor:Exception raised in Job[236]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)


MiniLM RAGAS (before alignment):
{'faithfulness': 0.9234, 'answer_relevancy': 0.3551, 'context_precision': 0.2642, 'context_recall': 0.3000}


Evaluating:   0%|          | 0/400 [00:00<?, ?it/s]

...

ERROR:ragas.executor:Exception raised in Job[48]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)
ERROR:ragas.executor:Exception raised in Job[104]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)
ERROR:ragas.executor:Exception raised in Job[236]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)


DistilUSE RAGAS (before alignment):
{'faithfulness': 0.9390, 'answer_relevancy': 0.3586, 'context_precision': 0.2642, 'context_recall': 0.3000}


Evaluating:   0%|          | 0/400 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[236]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)
ERROR:ragas.executor:Exception raised in Job[356]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)


DistilUSE RAGAS (after alignment):
{'faithfulness': 0.9704, 'answer_relevancy': 0.4259, 'context_precision': 0.3758, 'context_recall': 0.4500}


In [28]:
# Generation eval (EM/F1) using retrieved contexts
# Runs for all embedding models (before/after alignment).

import os
import re
import numpy as np
import pandas as pd
from typing import List

# LLM for answer generation
try:
    from langchain_openai import ChatOpenAI
except Exception as exc:
    raise RuntimeError("Install langchain-openai to run this cell.") from exc

gen_llm = ChatOpenAI(model=os.getenv("GEN_LLM_MODEL", "gpt-4.1-mini"), temperature=0, max_tokens=256, n=1)
GEN_SAMPLE_N = 50
GEN_TOP_K = 3

GEN_MODELS = [
    {"label": "MiniLM", "model_name": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", "query_prefix": "", "doc_prefix": ""},
    {"label": "BGE", "model_name": "BAAI/bge-m3", "query_prefix": "", "doc_prefix": ""},
    {"label": "IntFloat", "model_name": "intfloat/multilingual-e5-large", "query_prefix": "query: ", "doc_prefix": "passage: "},
    {"label": "LaBSE", "model_name": "sentence-transformers/LaBSE", "query_prefix": "", "doc_prefix": ""},
    {"label": "E5-Base", "model_name": "intfloat/multilingual-e5-base", "query_prefix": "query: ", "doc_prefix": "passage: "},
    {"label": "DistilUSE", "model_name": "sentence-transformers/distiluse-base-multilingual-cased-v2", "query_prefix": "", "doc_prefix": ""},
]


def normalize_text(s: str) -> str:
    s = s.lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def exact_match(pred: str, gold: str) -> float:
    return 1.0 if normalize_text(pred) == normalize_text(gold) else 0.0


def f1_score(pred: str, gold: str) -> float:
    pred_tokens = normalize_text(pred).split()
    gold_tokens = normalize_text(gold).split()
    if not pred_tokens and not gold_tokens:
        return 1.0
    if not pred_tokens or not gold_tokens:
        return 0.0
    common = {}
    for t in pred_tokens:
        common[t] = common.get(t, 0) + 1
    overlap = 0
    for t in gold_tokens:
        if common.get(t, 0) > 0:
            overlap += 1
            common[t] -= 1
    if overlap == 0:
        return 0.0
    precision = overlap / len(pred_tokens)
    recall = overlap / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


def build_retriever(model, *, query_prefix: str, doc_prefix: str):
    doc_df_local = context_pairs[["context_id", "context_squad"]].rename(columns={"context_squad": "doc_text"})
    doc_text_by_id = dict(zip(doc_df_local["context_id"], doc_df_local["doc_text"]))

    doc_emb = encode_texts(model, doc_df_local["doc_text"].tolist(), prefix=doc_prefix)
    index = faiss.IndexFlatIP(doc_emb.shape[1])
    index.add(doc_emb)

    ur_train_q = encode_texts(model, train_queries["question"].tolist(), prefix=query_prefix)
    train_doc_texts = [doc_text_by_id[cid] for cid in train_queries["context_id"].tolist()]
    en_train_d = encode_texts(model, train_doc_texts, prefix=doc_prefix)
    ur_train_q, en_train_d = filter_empty_vectors(ur_train_q, en_train_d)
    alignment = learn_procrustes_alignment(ur_train_q, en_train_d, center=True)

    return index, doc_df_local, alignment


def retrieve_contexts_for_model(
    query: str,
    *,
    model,
    index,
    doc_df_local: pd.DataFrame,
    alignment,
    query_prefix: str,
    k: int,
    use_alignment: bool,
) -> List[str]:
    qvec = encode_texts(model, [query], prefix=query_prefix)
    if use_alignment:
        qvec = apply_procrustes_alignment(qvec, alignment)
        qvec = safe_l2_normalize(qvec)
    scores, indices = index.search(qvec, min(k, index.ntotal))
    ctxs = []
    for idx in indices[0]:
        if idx < 0:
            continue
        ctxs.append(doc_df_local.iloc[idx]["doc_text"])
    return ctxs


def build_generation_dataset(
    *,
    model,
    index,
    doc_df_local: pd.DataFrame,
    alignment,
    query_prefix: str,
    use_alignment: bool,
) -> pd.DataFrame:
    sample = merged.dropna(subset=["question_uqa", "context_squad"]).sample(
        n=min(GEN_SAMPLE_N, len(merged)), random_state=42
    )
    rows = []
    for row in sample.itertuples(index=False):
        row = pd.Series(row._asdict())
        q = row.get("question_uqa")
        if not isinstance(q, str) or not q.strip():
            continue
        ctxs = retrieve_contexts_for_model(
            q,
            model=model,
            index=index,
            doc_df_local=doc_df_local,
            alignment=alignment,
            query_prefix=query_prefix,
            k=GEN_TOP_K,
            use_alignment=use_alignment,
        )
        gt = pick_answer(row)
        if not gt:
            continue
        rows.append({"question": q, "contexts": ctxs, "ground_truth": gt})
    return pd.DataFrame(rows)


def generate_answer(question: str, contexts: List[str]) -> str:
    context_block = "".join(f"[{i+1}] {c}" for i, c in enumerate(contexts))
    prompt = ("You are a QA system. Answer the question using ONLY the contexts. "
        "If the answer is not in the contexts, say 'I don't know'."
        f"Question: {question}"
        f"Contexts:{context_block}"
        "Answer:"
    )
    return gen_llm.invoke(prompt).content.strip()


def run_generation_eval(*, label: str, model, index, doc_df_local, alignment, query_prefix: str, use_alignment: bool) -> None:
    gen_df = build_generation_dataset(
        model=model,
        index=index,
        doc_df_local=doc_df_local,
        alignment=alignment,
        query_prefix=query_prefix,
        use_alignment=use_alignment,
    )
    if gen_df.empty:
        print(f"{label} Generation: no valid rows")
        return

    ems = []
    f1s = []
    for row in gen_df.itertuples(index=False):
        pred = generate_answer(row.question, row.contexts)
        ems.append(exact_match(pred, row.ground_truth))
        f1s.append(f1_score(pred, row.ground_truth))

    tag = "after" if use_alignment else "before"
    print(f"{label} Generation ({tag} alignment): em={float(np.mean(ems)):.4f}, f1={float(np.mean(f1s)):.4f}")


for cfg in GEN_MODELS:
    model = get_model(cfg["model_name"])
    index, doc_df_local, alignment = build_retriever(
        model,
        query_prefix=cfg["query_prefix"],
        doc_prefix=cfg["doc_prefix"],
    )
    run_generation_eval(
        label=cfg["label"],
        model=model,
        index=index,
        doc_df_local=doc_df_local,
        alignment=alignment,
        query_prefix=cfg["query_prefix"],
        use_alignment=False,
    )
    run_generation_eval(
        label=cfg["label"],
        model=model,
        index=index,
        doc_df_local=doc_df_local,
        alignment=alignment,
        query_prefix=cfg["query_prefix"],
        use_alignment=True,
    )


MiniLM Generation (before alignment): em=0.1800, f1=0.2212
MiniLM Generation (after alignment): em=0.1800, f1=0.2078
BGE Generation (before alignment): em=0.3200, f1=0.3989
BGE Generation (after alignment): em=0.2800, f1=0.3312
IntFloat Generation (before alignment): em=0.3200, f1=0.3933
IntFloat Generation (after alignment): em=0.3000, f1=0.3478
LaBSE Generation (before alignment): em=0.0400, f1=0.0400
LaBSE Generation (after alignment): em=0.1400, f1=0.1684
E5-Base Generation (before alignment): em=0.1200, f1=0.1617
E5-Base Generation (after alignment): em=0.2800, f1=0.3080
DistilUSE Generation (before alignment): em=0.0800, f1=0.0818
DistilUSE Generation (after alignment): em=0.1600, f1=0.1600
